## ⚠️ Note

This notebook is a cleaned and modularized version of experiments
conducted in Google Colab. Paths and configurations are simplified
for clarity and documentation purposes.

Some training steps are computationally expensive and may not be
reproducible without the original dataset and GPU resources.


In [ ]:
# ==============================
# Setup: Download LoRA Training Script
# ==============================

# This script is provided by Hugging Face Diffusers
# and is required to train LoRA using Stable Diffusion

import os

if not os.path.exists("train_text_to_image_lora.py"):
    print("⬇️ Downloading LoRA training script...")
    !wget -q -O train_text_to_image_lora.py \
    https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py
else:
    print("✅ LoRA training script already exists")



In [ ]:
# ==============================
# Train LoRA for Batik Style
# ==============================
import json
from IPython import get_ipython
import os

def train_lora_batik(
    batik_name: str,
    epochs: int = 50,
    batch_size: int = 1,
    learning_rate: float = 1e-4,
):
    """
    Train LoRA for a specific batik style using Stable Diffusion.
    """

    print(f"\n🚀 Starting LoRA training for batik style: {batik_name}")

    # ==============================
    # Base configuration
    # ==============================
    BASE_MODEL = "runwayml/stable-diffusion-v1-5"
    DATASET_DIR = "..."
    OUTPUT_DIR = f".../{batik_name}"

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("📁 Dataset directory:", DATASET_DIR)
    print("📁 Output directory :", OUTPUT_DIR)

    # ==============================
    # Generate metadata.jsonl
    # ==============================
    metadata = []

    for file in os.listdir(DATASET_DIR):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            metadata.append({
                "file_name": file,
                "text": f"batik_style_{batik_name}"
            })

    metadata_path = os.path.join(DATASET_DIR, "metadata.jsonl")
    with open(metadata_path, "w") as f:
        for item in metadata:
            json.dump(item, f)
            f.write("\n")

    print(f"✅ metadata.jsonl generated with {len(metadata)} samples")

    # ==============================
    # Training command (Diffusers LoRA)
    # ==============================
    command = (
        f"accelerate launch train_text_to_image_lora.py "
        f"--pretrained_model_name_or_path={BASE_MODEL} "
        f"--train_data_dir={DATASET_DIR} "
        f"--caption_column=text "
        f"--resolution=512 "
        f"--train_batch_size={batch_size} "
        f"--gradient_accumulation_steps=4 "
        f"--num_train_epochs={epochs} "
        f"--learning_rate={learning_rate} "
        f"--output_dir={OUTPUT_DIR} "
    )

    print("\n🧠 Training command:")
    print(command)

    # ==============================
    # Execute training
    # ==============================
    get_ipython().system(command)

    print(f"\n🎉 Training completed! LoRA saved in: {OUTPUT_DIR}")


In [ ]:
train_lora_batik(
    batik_name="megamendung",
    epochs=50,
    batch_size=1,
    learning_rate=1e-4
)